# Radius_arcsec -> Radius_kpc: D_A from z vs. D implied by the data

For each galaxy: compute `D_A(z)` from scratch (the paper's stated cosmology,
H0=70, Om0=0.3), and separately back out the distance implied by the
catalogue's own `Radius_arcsec` -> `Radius_kpc` conversion:

```
D_implied = Radius_kpc / Radius_arcsec_in_radians
```

If `Radius_kpc` really was built from `D_A(z)` at H0=70, these two should
match to within rounding. This is the table to send back with "what distance
did you use here?" 

In [ ]:
import numpy as np
import pandas as pd
from astropy.cosmology import FlatLambdaCDM
import astropy.units as apu

DATA_FILE = "rar_with_ml_direct_phot.csv"
H0 = 70.0
OM0 = 0.3

df = pd.read_csv(DATA_FILE)
cosmo = FlatLambdaCDM(H0=H0, Om0=OM0)
print(f"Loaded {len(df)} rows, {df['Galaxy'].nunique()} galaxies")
print(f"Cosmology assumed: FlatLambdaCDM(H0={H0}, Om0={OM0})")

In [ ]:
rows = []
for galaxy, g in df.groupby("Galaxy"):
    z = float(g["z"].iloc[0])

    D_A_Mpc = cosmo.angular_diameter_distance(z).to(apu.Mpc).value

    radius_rad = (g["Radius_arcsec"].to_numpy() * apu.arcsec).to(apu.rad).value
    D_implied_per_point = (g["Radius_kpc"].to_numpy() / radius_rad) / 1000.0  # Mpc
    D_implied_Mpc = float(np.median(D_implied_per_point))
    D_implied_std_Mpc = float(np.std(D_implied_per_point))
    c = 299792.458  # km/s
    D_hubble_law = c*z / H0  # Mpc, approximate for low z

    rows.append({
        "Galaxy": galaxy,
        "z": z,
        "D_A_Mpc (from z, H0=70)": D_A_Mpc,
        "D_implied_Mpc (from Radius_kpc/Radius_arcsec)": D_implied_Mpc,
        "D_implied_std_Mpc": D_implied_std_Mpc,
        "D_hubble_law_Mpc": D_hubble_law,
        "percent_difference": 100.0 * (D_implied_Mpc - D_A_Mpc) / D_A_Mpc,

    })

table = pd.DataFrame(rows).sort_values("z").reset_index(drop=True)

OUT_FILE = "distance_check_simple.csv"
table.to_csv(OUT_FILE, index=False)
print(f"Saved {len(table)} galaxies to {OUT_FILE}")
print(f"Median percent difference across all {len(table)} galaxies: {table['percent_difference'].median():.2f}%")

with pd.option_context("display.max_rows", None, "display.width", 140, "display.float_format", "{:.4f}".format):
    print()
    print(table.to_string(index=False))

## Distance-uncertainty propagation: sigma_z -> sigma_D, for every galaxy

The pipeline currently uses a flat placeholder for the distance-prior width,
`e_d = 0.15 * D(z)` (`DISTANCE_PRIOR_FRAC_WIDTH` in `dm_likelihood.py`) -- not
derived from any actual redshift/distance uncertainty.

Here we propagate an assumed redshift uncertainty `SIGMA_Z` through the same
`D_A(z)` used above, via `sigma_D ~= |dD/dz| * SIGMA_Z` (finite difference on
the real cosmology, not the low-z linear approximation), for **every**
galaxy, and compare it directly to the 15% placeholder.

`SIGMA_Z = 0.01` below is a flat, z-independent placeholder standing in for
whatever the data provider eventually reports -- replace it once a real
number (a fixed instrumental sigma_z, a fractional sigma_z/z, or a
peculiar-velocity sigma_pec) is available.

In [ ]:
SIGMA_Z = 0.01  # flat/z-independent placeholder -- replace once the data provider gives a real number
DISTANCE_PRIOR_FRAC_WIDTH = 0.15  # matches dm_likelihood.py's current placeholder
dz_step = 1e-5  # finite-difference step -- small vs. SIGMA_Z and vs. the sample's own z values

def D_A_kpc(z):
    return cosmo.angular_diameter_distance(z).to(apu.kpc).value

rows_unc = []
for galaxy, g in df.groupby("Galaxy"):
    z = float(g["z"].iloc[0])
    D0_kpc = D_A_kpc(z)
    dDdz = (D_A_kpc(z + dz_step) - D_A_kpc(z - dz_step)) / (2 * dz_step)
    sigma_D_propagated = abs(dDdz) * SIGMA_Z
    e_d_placeholder = DISTANCE_PRIOR_FRAC_WIDTH * D0_kpc

    rows_unc.append({
        "Galaxy": galaxy,
        "z": z,
        "D(z) kpc": D0_kpc,
        "dD/dz (kpc)": dDdz,
        "sigma_D propagated (kpc)": sigma_D_propagated,
        "sigma_D / D(z) [%]": 100.0 * sigma_D_propagated / D0_kpc,
        "15% placeholder e_d (kpc)": e_d_placeholder,
        "placeholder / propagated": e_d_placeholder / sigma_D_propagated,
        "sigma_z / z": SIGMA_Z / z,
    })

unc_table = pd.DataFrame(rows_unc).sort_values("z").reset_index(drop=True)

OUT_FILE_UNC = "distance_uncertainty_propagation.csv"
unc_table.to_csv(OUT_FILE_UNC, index=False)

n_breakdown = int((unc_table["sigma_z / z"] > 0.3).sum())
print(f"Saved {len(unc_table)} galaxies to {OUT_FILE_UNC}")
print(f"SIGMA_Z={SIGMA_Z} exceeds 30% of z for {n_breakdown}/{len(unc_table)} galaxies")
print("-- the linear sigma_D ~= |dD/dz|*sigma_z approximation is questionable for those (sigma_z is not small vs. z)")
print()
print(f"Median placeholder/propagated ratio: {unc_table['placeholder / propagated'].median():.2f}")
print(f"Min z: {unc_table['z'].min():.4f}, sigma_D propagated there: {unc_table['sigma_D propagated (kpc)'].iloc[0]:.1f} kpc "
      f"vs. D(z)={unc_table['D(z) kpc'].iloc[0]:.1f} kpc")

with pd.option_context("display.max_rows", None, "display.width", 160, "display.float_format", "{:.4f}".format):
    print()
    print(unc_table.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.plot(unc_table["z"], unc_table["sigma_D propagated (kpc)"], "o-", label=fr"propagated ($\sigma_z$={SIGMA_Z})")
ax.plot(unc_table["z"], unc_table["15% placeholder e_d (kpc)"], "s--", label="15% placeholder")
ax.set_xlabel("z")
ax.set_ylabel(r"$e_d$ (kpc)")
ax.set_title("Distance-prior width: propagated vs. placeholder")
ax.legend()

ax = axes[1]
ax.plot(unc_table["z"], unc_table["placeholder / propagated"], "o-", color="C2")
ax.axhline(1.0, color="k", lw=0.8, ls=":")
ax.set_xlabel("z")
ax.set_ylabel("placeholder / propagated")
ax.set_title("Where the 15% placeholder over/understates\nthe propagated uncertainty")

fig.tight_layout()
fig.savefig("distance_uncertainty_propagation.png", dpi=150)
plt.show()
print("Saved distance_uncertainty_propagation.png")